In [1]:
import sys
import os

sys.path.append(os.path.dirname(os.getcwd()))

from modules.loader import load_document
from modules.splitter import split_documents

import warnings
warnings.filterwarnings("ignore")

c:\Users\Personal\Documents\python-hands0n\myenv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.6.3) or chardet (7.4.3)/charset_normalizer (3.4.7) doesn't match a supported version!
  warnings.warn(


In [2]:
sample_txt = "../data/sample_docs/sample_text.txt"

if not os.path.exists(sample_txt):
    os.makedirs(os.path.dirname(sample_txt), exist_ok = True)
    with open(sample_txt, "w", encoding = "utf-8") as f:
        f.write("This is a short sample docuemnt about LangChain.\n" * 50)

print("=== Loading TXT ===")
docs = load_document(sample_txt)
print("Loaded documents:", len(docs))
print("First 200 chars: ", docs[0].page_content[:200])

=== Loading TXT ===
Loaded documents: 1
First 200 chars:  scikit-learn (formerly scikits.learn and also known as sklearn) is a free and open-source machine learning library for the Python programming language.[1] It features various classification, regressio


In [3]:
print("=== Splitting into chunks ===")
chunks = split_documents(docs, chunk_size = 200, chunk_overlap = 20)

print("Total chunks: ", len(chunks))
print(f"Chunk 0 (first 200 chars):\n{chunks[0].page_content[:200]}")
print(f"Chunk 0 metadata: {chunks[0].metadata}")

=== Splitting into chunks ===
Total chunks:  45
Chunk 0 (first 200 chars):
scikit-learn (formerly scikits.learn and also known as sklearn) is a free and open-source machine learning library for the Python programming language.[1] It features various classification,
Chunk 0 metadata: {'source': 'c:\\Users\\Personal\\Documents\\python-hands0n\\data\\sample_docs\\sample_text.txt', 'chunk_index': 0}


In [4]:
# import modules.utils as utils

In [5]:
# from modules.utils import get_embedding_model

# for provider in ["openai", "gemini"]:
#     try:
#         print(f"=== Testing {provider.upper()} ===")
#         model = get_embedding_model(provider)
#         print("Model loaded: ", type(model))
#     except Exception as e:
#         print(e)

In [6]:
from modules.embedder import build_or_update_vectorstore
import json

# load config
with open("../config/config.json", "r") as f:
    config = json.load(f)

# load and split documents
sample_txt = "../data/sample_docs/sample_text.txt"
docs = load_document(sample_txt)
chunks = split_documents(docs, chunk_size = 200, chunk_overlap = 40)

Loaded .env from: c:\Users\Personal\Documents\python-hands0n\config\.env


ImportError: cannot import name 'Pinecone' from 'pinecone' (c:\Users\Personal\Documents\python-hands0n\myenv\Lib\site-packages\pinecone\__init__.py)

In [ ]:
# build vectorstore
vectorstore = build_or_update_vectorstore(chunks, config)

🔹 Embedding Provider: openai
🔹 Vector Store Type: pinecone
Loading OpenAI embedding model...
✅ Embedding model loaded successfully.
☁️ Using Pinecone vector store...
🔍 Detecting embedding dimension...
📏 Embedding dimension detected: 1536


MaxRetryError: HTTPSConnectionPool(host='controller.us-east-1.pinecone.io', port=443): Max retries exceeded with url: /databases (Caused by NameResolutionError("HTTPSConnection(host='controller.us-east-1.pinecone.io', port=443): Failed to resolve 'controller.us-east-1.pinecone.io' ([Errno 11001] getaddrinfo failed)"))

In [ ]:
# Test a similarity search
query = "Explain scikit-learn in short?"
results = vectorstore.similarity_search(query, k = 2)

print(f"Query: {query}\n\n")
for i, doc in enumerate(results, 1):
    print(f"\n{i}: {doc.page_content[:100]}")


AttributeError: 'NoneType' object has no attribute 'similarity_search'

In [ ]:
from modules.retriever import get_retriever

retriever = get_retriever(vectorstore, config) # either base or multiquery

query = "Explain scikit-learn in short?"

print(f"Query: {query}\n")

results = retriever.invoke(query)
for i, doc in enumerate(results, 1):
    print(f"\n{i}: {doc.page_content[:100]}")

Using base retriever(simple similarity search)
Query: Explain scikit-learn in short?


1: scikit-learn (formerly scikits.learn and also known as sklearn) is a free and open-source machine le

2: scikit-learn (formerly scikits.learn and also known as sklearn) is a free and open-source machine le

3: scikit-learn (formerly scikits.learn and also known as sklearn) is a free and open-source machine le


In [ ]:
from modules.rag_chain import build_rag_chain
from modules.memory import add_memory_to_chain

session_id = "test_user_001"

# use the rag chain
rag_chain = build_rag_chain(retriever, config)
rag_with_memory = add_memory_to_chain(rag_chain, session_id, enabled = True)

print("\n💬 First question:")
resp1 = rag_with_memory.invoke(
    {"question": "Give detailed explanation on scikit-learn."},
    config={"configurable": {"session_id": session_id}},
)
print(resp1.content)

print("\n💬 Follow-up question:")
resp2 = rag_with_memory.invoke(
    {"question": "Can you summarize our entire conversation so far in 3 bullet points?"},
    config={"configurable": {"session_id": session_id}},
)
print(resp2.content)

🔹 Building RAG chain using provider: openai
✅ Hybrid RAG + Memory chain built successfully using OPENAI.
🟢 Hybrid Memory ON for session: test_user_001

💬 First question:
Scikit-learn is a powerful and widely-used machine learning library in Python that provides a range of tools for data analysis and modeling. Here’s a detailed explanation of its key features and functionalities:

### 1. **Core Features:**
   - **Algorithms:** Scikit-learn includes a variety of supervised and unsupervised learning algorithms. This includes classification algorithms (like SVM, decision trees, and random forests), regression algorithms (like linear regression and ridge regression), clustering algorithms (like k-means and DBSCAN), and dimensionality reduction techniques (like PCA).
   
   - **Preprocessing:** The library offers various preprocessing techniques to prepare data for modeling. This includes scaling (standardization and normalization), encoding categorical variables, handling missing values, an